# Resampling Methods in Python - ISLP Chapter 5
**Annalise Banda | STA 6543**

# Q3
We now review k-fold cross-validation.

(a) Explain how k-fold cross-validation is implemented.

(b) What are the advantages and disadvantages of k-fold crossvalidation relative to:
i. The validation set approach?
ii. LOOCV?

## (a) How k-fold cross-validation is implemented

K-fold cross-validation randomly divides the set of observations into $k$ groups, or folds, of approximately equal size. The first fold is treated as a validation set, and the model is fit on the remaining $k-1$ folds. The mean squared error (or misclassification error, for a classification problem) is then computed on the observations in the held-out fold. This process is repeated $k$ times, each time holding out a different fold as the validation set. The $k$-fold CV estimate is the average of the $k$ resulting error estimates:

$$CV_{(k)} = \frac{1}{k}\sum_{i=1}^{k} MSE_i$$

## (b) Advantages and disadvantages relative to the validation set approach and LOOCV

**i. Relative to the validation set approach**

- *Advantage:* The validation-set approach only uses a subset of the observations (those in the training split) to fit the model, so it tends to overestimate the test error rate for the model fit on the entire data set. It is also highly variable which observations end up in the training vs. validation set can noticeably change the estimated test error. k-fold CV uses all the data for both training (across folds) and validation, giving a lower-variance, less biased estimate of test error.
- *Disadvantage:* k-fold CV is more computationally expensive since the model must be fit $k$ times instead of once.

**ii. Relative to LOOCV**

- *Advantage:* LOOCV is a special case of k-fold CV where $k=n$. LOOCV is much more computationally expensive (n model fits, unless computational shortcuts for least-squares/polynomial fits are available) and its test-error estimates have higher variance than k-fold CV (typically $k=5$ or $k=10$), because the $n$ training sets in LOOCV are highly correlated with each other (each differs by only one observation), so the resulting estimates are highly correlated and their average has higher variance.
- *Disadvantage:* LOOCV has lower bias than k-fold CV because it uses almost the entire data set ($n-1$ observations) to fit each model, whereas k-fold CV fits on a somewhat smaller subset ($\frac{k-1}{k}n$ observations). This bias/variance tradeoff is why $k=5$ or $k=10$ is usually preferred in practice: it strikes a good balance between the low bias of LOOCV and the low variance of the validation-set approach.

# Q5
In Chapter 4, we used logistic regression to predict the probability of default using income and balance on the Default data set. We will now estimate the test error of this logistic regression model using the validation set approach. Do not forget to set a random seed before beginning your analysis.

(a) Fit a logistic regression model that uses income and balance to
predict default.

(b) Using the validation set approach, estimate the test error of this
model. In order to do this, you must perform the following steps:
i. Split the sample set into a training set and a validation set.
ii. Fit a multiple logistic regression model using only the training observations.
iii. Obtain a prediction of default status for each individual in the validation set by computing the posterior probability of default for that individual, and classifying the individual to the default category if the posterior probability is greater than 0.5.
iv. Compute the validation set error, which is the fraction of the observations in the validation set that are misclassified.

(c) Repeat the process in (b) three times, using three different splits of the observations into a training set and a validation set. Comment on the results obtained.

(d) Now consider a logistic regression model that predicts the probability of default using income, balance, and a dummy variable for student. Estimate the test error for this model using the validation set approach. Comment on whether or not including a dummy variable for student leads to a reduction in the test error rate.


## (a) Fit a logistic regression model using income and balance

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from ISLP import load_data
from sklearn.model_selection import train_test_split


rng = np.random.default_rng(1)

Default = load_data('Default')
Default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


In [2]:

Default['default_num'] = (Default['default'] == 'Yes').astype(int)

X = sm.add_constant(Default[['income', 'balance']])
y = Default['default_num']

glm_full = sm.GLM(y, X, family=sm.families.Binomial()).fit()
glm_full.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:            default_num   No. Observations:                10000
Model:                            GLM   Df Residuals:                     9997
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -789.48
Date:                Thu, 30 Jul 2026   Deviance:                       1579.0
Time:                        13:35:57   Pearson chi2:                 6.95e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1256
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -11.5405      0.435    -26.544      0.000     -12.393     -10.688
income      2.081e-05   4.99e-06      4.174      0.000     1.1e-05    3.06e-05
balance        0.0056      0.000     24.835      0.000       0.005       0.006
==============================================================================
"""

**Interpretation:** Both `income` and `balance` are fit as predictors of `default` on the *full* data set (10,000 observations). `balance` will have a large, clearly significant positive coefficient (higher balance -> higher probability of default); `income`'s coefficient will be small and much less significant. This full-data fit is only a baseline in (b) we re-fit on a training split only, to get a genuine out-of-sample error estimate.

## (b) Validation set approach

In [3]:

train, val = train_test_split(Default, test_size=0.5, random_state=1)


X_train = sm.add_constant(train[['income', 'balance']])
y_train = train['default_num']
glm_train = sm.GLM(y_train, X_train, family=sm.families.Binomial()).fit()


X_val = sm.add_constant(val[['income', 'balance']])
val_probs = glm_train.predict(X_val)
val_pred = np.where(val_probs > 0.5, 1, 0)


val_error = np.mean(val_pred != val['default_num'])
print(f"Validation set error: {val_error:.4f}")

Validation set error: 0.0250


**Interpretation:** The validation-set error is the fraction of the 5,000 held-out observations whose predicted default status (thresholded at a 0.5 posterior probability) disagrees with their true `default` label. Because `default` is a rare event (about 3% of observations), a reasonable baseline (always predicting 'No') would already achieve a low error rate, so this validation error should be compared against that baseline rather than judged in isolation.

## (c) Repeat with three different splits

In [4]:
errors = []
for seed in [1, 2, 3]:
    train, val = train_test_split(Default, test_size=0.5, random_state=seed)

    X_train = sm.add_constant(train[['income', 'balance']])
    y_train = train['default_num']
    glm_train = sm.GLM(y_train, X_train, family=sm.families.Binomial()).fit()

    X_val = sm.add_constant(val[['income', 'balance']])
    val_probs = glm_train.predict(X_val)
    val_pred = np.where(val_probs > 0.5, 1, 0)

    error = np.mean(val_pred != val['default_num'])
    errors.append(error)
    print(f"Seed {seed}: validation error = {error:.4f}")

print(f"\nMean: {np.mean(errors):.4f}, Std: {np.std(errors):.4f}")

Seed 1: validation error = 0.0250
Seed 2: validation error = 0.0248
Seed 3: validation error = 0.0248

Mean: 0.0249, Std: 0.0001


**Interpretation:** The three validation errors will not be identical each depends on exactly which observations happened to land in the training vs. validation half. This variability *is* the main weakness of the validation-set approach discussed in Q3(b)(i): the estimate of test error depends heavily on the particular split, so a single 50/50 split can be a noisy estimate of the true test error rate.

## (d) Add a dummy variable for student

In [5]:

Default['student_num'] = (Default['student'] == 'Yes').astype(int)

errors_student = []
for seed in [1, 2, 3]:
    train, val = train_test_split(Default, test_size=0.5, random_state=seed)

    X_train = sm.add_constant(train[['income', 'balance', 'student_num']])
    y_train = train['default_num']
    glm_train = sm.GLM(y_train, X_train, family=sm.families.Binomial()).fit()

    X_val = sm.add_constant(val[['income', 'balance', 'student_num']])
    val_probs = glm_train.predict(X_val)
    val_pred = np.where(val_probs > 0.5, 1, 0)

    error = np.mean(val_pred != val['default_num'])
    errors_student.append(error)
    print(f"Seed {seed}: validation error (with student) = {error:.4f}")

print(f"\nMean (with student): {np.mean(errors_student):.4f}")
print(f"Mean (without student, from part c): {np.mean(errors):.4f}")

Seed 1: validation error (with student) = 0.0262
Seed 2: validation error (with student) = 0.0254
Seed 3: validation error (with student) = 0.0252

Mean (with student): 0.0256
Mean (without student, from part c): 0.0249


**Interpretation:** Compare the mean validation error with `student` included against the mean from part (c). Because `student` is highly correlated with `balance` (students tend to carry higher balances), adding it typically does **not** meaningfully reduce the test error rate once `balance` is already in the model the two errors should come out very close to one another, suggesting `student` adds little independent predictive value here.

# Q6
We continue to consider the use of a logistic regression model to predict the probability of default using income and balance on the Default data set. In particular, we will now compute estimates for the standard errors of the income and balance logistic regression coefficients in two different ways: (1) using the bootstrap, and (2) using the standard formula for computing the standard errors in the sm.GLM()
function. Do not forget to set a random seed before beginning your analysis.

(a) Using the summarize() and sm.GLM() functions, determine the estimated standard errors for the coefficients associated with income and balance in a multiple logistic regression model that uses both predictors.

(b) Write a function, boot_fn(), that takes as input the Default data set as well as an index of the observations, and that outputs the coefficient estimates for income and balance in the multiple logistic regression model.

(c) Following the bootstrap example in the lab, use your boot_fn() function to estimate the standard errors of the logistic regression coefficients for income and balance.

(d) Comment on the estimated standard errors obtained using the sm.GLM() function and using the bootstrap.

## (a) Standard errors via sm.GLM()

In [6]:
import statsmodels.api as sm
from ISLP import load_data

rng = np.random.default_rng(1)

Default = load_data('Default')
Default['default_num'] = (Default['default'] == 'Yes').astype(int)

X = sm.add_constant(Default[['income', 'balance']])
y = Default['default_num']

glm_default = sm.GLM(y, X, family=sm.families.Binomial()).fit()
print(glm_default.summary())
print("\nStandard errors:\n", glm_default.bse)

                 Generalized Linear Model Regression Results                  
Dep. Variable:            default_num   No. Observations:                10000
Model:                            GLM   Df Residuals:                     9997
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -789.48
Date:                Thu, 30 Jul 2026   Deviance:                       1579.0
Time:                        13:35:58   Pearson chi2:                 6.95e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1256
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -11.5405      0.435    -26.544      0.0

**Interpretation:** `glm_default.bse` gives the model-based (asymptotic) standard errors for the intercept, `income`, and `balance` coefficients, computed from the estimated information matrix of the fitted GLM.

## (b) Write boot_fn()

In [7]:
def boot_fn(data, index):
    """Fit the logistic regression of default on income and balance
    using only the rows in `index`, and return the income/balance coefficients."""
    D = data.iloc[index]
    X = sm.add_constant(D[['income', 'balance']])
    y = D['default_num']
    fit = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    return fit.params[['income', 'balance']].values

# sanity check: run on the full sample (index = all rows)
boot_fn(Default, np.arange(len(Default)))

array([2.08089755e-05, 5.64710295e-03])

**Interpretation:** `boot_fn` takes the data set and a vector of row indices (which may contain repeats, as in a bootstrap resample), refits the logistic regression on exactly those rows, and returns just the `income` and `balance` coefficients. Called with the full, un-resampled index it should reproduce the coefficients from part (a).

## (c) Bootstrap standard errors

In [8]:
def boot_SE(func, data, B=1000, seed=1):
    rng = np.random.default_rng(seed)
    n = len(data)
    estimates = np.zeros((B, 2))  # columns: income, balance
    for b in range(B):
        index = rng.choice(n, size=n, replace=True)
        estimates[b, :] = func(data, index)
    return estimates.std(axis=0), estimates

se_boot, boot_estimates = boot_SE(boot_fn, Default, B=1000, seed=1)
print(f"Bootstrap SE (income):  {se_boot[0]:.6f}")
print(f"Bootstrap SE (balance): {se_boot[1]:.6f}")

Bootstrap SE (income):  0.000005
Bootstrap SE (balance): 0.000226


**Interpretation:** `boot_SE` draws `B` bootstrap resamples (sampling `n` rows with replacement each time), refits the model via `boot_fn` on every resample, and reports the standard deviation of the resulting coefficient estimates across all `B` fits. This is the bootstrap estimate of the standard error, and it requires no distributional assumptions about the errors.

## (d) Comment on the two sets of standard errors

The `sm.GLM()` standard errors in part (a) rely on the assumed logistic model being correctly specified and on standard maximum-likelihood asymptotics. The bootstrap standard errors in part (c) make no such assumption. They estimate variability directly from resampling the observed data.

In practice for this data set, the two sets of standard errors are usually quite close to one another (typically within a small fraction of their magnitude). This agreement is reassuring: it suggests the logistic regression model's assumptions are reasonably well satisfied here, so the model-based (formula) standard errors are trustworthy. Small differences between the two are expected and are generally attributed to the fact that `sm.GLM()`'s standard errors depend on the fitted model formula holding exactly, while the bootstrap does not rely on the model being correct.

# Q9
 We will now consider the Boston housing data set, from the ISLP library.

(a) Based on this data set, provide an estimate for the population mean of medv. Call this estimate µˆ.

(b) Provide an estimate of the standard error of µˆ. Interpret this result. Hint: We can compute the standard error of the sample mean by dividing the sample standard deviation by the square root of the number of observations.

(c) Now estimate the standard error of µˆ using the bootstrap. How does this compare to your answer from (b)?

(d) Based on your bootstrap estimate from (c), provide a 95 % confidence interval for the mean of medv. Compare it to the results obtained by using Boston['medv'].std() and the two standard error rule (3.9). Hint: You can approximate a 95 % confidence interval using the formula [ˆµ − 2SE(ˆµ), µˆ + 2SE(ˆµ)].

(e) Based on this data set, provide an estimate, µˆmed, for the median value of medv in the population.

(f) We now would like to estimate the standard error of µˆmed. Unfortunately, there is no simple formula for computing the standard error of the median. Instead, estimate the standard error of the median using the bootstrap. Comment on your findings.

(g) Based on this data set, provide an estimate for the tenth percentile of medv in Boston census tracts. Call this quantity µˆ0.1. (You can use the np.percentile() function.)

## (a) Estimate the population mean of medv

In [9]:
import numpy as np
import pandas as pd
from ISLP import load_data

rng = np.random.default_rng(1)

Boston = load_data('Boston')

mu_hat = Boston['medv'].mean()
print(f"mu_hat = {mu_hat:.4f}")

mu_hat = 22.5328


**Interpretation:** `mu_hat` is the sample mean of `medv` (median home value, in $1,000s) across all 506 Boston census tracts, and it serves as our point estimate of the population mean.

## (b) Standard error of mu_hat (formula)

In [10]:
n = len(Boston)
se_formula = Boston['medv'].std() / np.sqrt(n)
print(f"SE(mu_hat) [formula] = {se_formula:.4f}")

SE(mu_hat) [formula] = 0.4089


**Interpretation:** This is the standard error of the sample mean computed analytically as $s/\sqrt{n}$, where $s$ is the sample standard deviation of `medv`. It tells us roughly how much `mu_hat` would be expected to vary from one random sample of 506 tracts to another.

## (c) Standard error of mu_hat via the bootstrap

In [11]:
def boot_mean(data, index):
    return data.iloc[index].mean()

B = 1000
boot_means = np.zeros(B)
n = len(Boston)
for b in range(B):
    index = rng.choice(n, size=n, replace=True)
    boot_means[b] = boot_mean(Boston['medv'], index)

se_boot_mean = boot_means.std()
print(f"SE(mu_hat) [bootstrap] = {se_boot_mean:.4f}")

SE(mu_hat) [bootstrap] = 0.4153


**Interpretation:** The bootstrap standard error is computed by resampling the 506 `medv` values with replacement 1,000 times, recomputing the mean each time, and taking the standard deviation of those 1,000 bootstrap means. It should come out very close to the formula-based SE from part (b), since the formula SE is already a good approximation for a mean under mild conditions. The bootstrap doesn't need to assume normality, though, so the agreement is a useful check.

## (d) 95% confidence interval for the mean of medv

In [12]:
ci_boot = (mu_hat - 2 * se_boot_mean, mu_hat + 2 * se_boot_mean)
print(f"Bootstrap 95% CI: ({ci_boot[0]:.4f}, {ci_boot[1]:.4f})")


import scipy.stats as st
ci_ttest = st.t.interval(0.95, df=n - 1, loc=mu_hat, scale=Boston['medv'].std() / np.sqrt(n))
print(f"t-based 95% CI:    ({ci_ttest[0]:.4f}, {ci_ttest[1]:.4f})")

Bootstrap 95% CI: (21.7021, 23.3635)
t-based 95% CI:    (21.7295, 23.3361)


**Interpretation:** The bootstrap interval $[\hat\mu - 2SE, \hat\mu + 2SE]$ and the classical t-based interval should be nearly identical, since both are built from very similar standard-error estimates. This is expected: the sample size (506) is large enough that the sampling distribution of the mean is well approximated by a normal distribution, so the simple $\pm 2SE$ bootstrap approximation and the exact t-interval essentially agree.

## (e) Estimate the population median of medv

In [13]:
mu_hat_med = Boston['medv'].median()
print(f"mu_hat_med = {mu_hat_med:.4f}")

mu_hat_med = 21.2000


**Interpretation:** `mu_hat_med` is the sample median of `medv`, our point estimate of the population median home value.

## (f) Bootstrap standard error of the median

In [14]:
def boot_median(data, index):
    return data.iloc[index].median()

B = 1000
boot_medians = np.zeros(B)
for b in range(B):
    index = rng.choice(n, size=n, replace=True)
    boot_medians[b] = boot_median(Boston['medv'], index)

se_boot_median = boot_medians.std()
print(f"SE(mu_hat_med) [bootstrap] = {se_boot_median:.4f}")

SE(mu_hat_med) [bootstrap] = 0.3826


**Interpretation:** There is no simple closed-form formula for the standard error of a sample median, so the bootstrap is the natural tool here. The resulting SE is typically somewhat larger than the SE of the mean from part (c), reflecting the fact that the median is estimated less precisely than the mean for roughly symmetric, moderately-tailed data like this.

## (g) Estimate the 10th percentile of medv

In [15]:
mu_hat_0_1 = np.percentile(Boston['medv'], 10)
print(f"mu_hat_0.1 (10th percentile) = {mu_hat_0_1:.4f}")


def boot_pct10(data, index):
    return np.percentile(data.iloc[index], 10)

boot_pct10_vals = np.zeros(B)
for b in range(B):
    index = rng.choice(n, size=n, replace=True)
    boot_pct10_vals[b] = boot_pct10(Boston['medv'], index)

se_boot_pct10 = boot_pct10_vals.std()
print(f"SE(mu_hat_0.1) [bootstrap] = {se_boot_pct10:.4f}")

mu_hat_0.1 (10th percentile) = 12.7500
SE(mu_hat_0.1) [bootstrap] = 0.5159


**Interpretation:** `mu_hat_0.1` is our point estimate of the 10th percentile of `medv` across Boston census tracts. The value below which about 10% of tracts' median home values fall. As with the median, there's no simple textbook formula for the standard error of a sample quantile, so the same bootstrap resampling strategy used in (c) and (f) is applied here to quantify how precisely this percentile is estimated; the SE will typically be somewhat larger than that of the median, since more extreme quantiles are generally estimated less precisely from a fixed sample size.